# 05  -  Baseline Models


In [1]:
exec(open('00_config.py').read())

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (classification_report, roc_auc_score,
                              precision_score, recall_score,
                              precision_recall_curve, auc, f1_score,
                              matthews_corrcoef)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import torch, torch.nn as nn

cc = pd.read_csv(CC_FEATURES_CSV)
mm = pd.read_csv(MM_FEATURES_CSV)
hi = pd.read_csv(HI_FEATURES_CSV)
print(f'CC={cc.shape}  MM={mm.shape}  HI={hi.shape}')

Config loaded. BASE_DIR: /home/compute.ashesi.lan/sedem.agudetse/UniFraud-GH
CC=(1852394, 15)  MM=(550357, 10)  HI=(2655, 26)


## 1. Helpers

In [2]:
def find_best_threshold(probs, y_true):
    """MCC-optimised when fraud rate <5%, F1 otherwise."""
    use_mcc = y_true.mean() < 0.05
    best_t, best_s = 0.5, -999
    for t in np.arange(0.01, 0.99, 0.005):
        preds = (probs >= t).astype(int)
        if preds.sum() == 0 or preds.sum() == len(preds):
            continue
        s = matthews_corrcoef(y_true, preds) if use_mcc else f1_score(y_true, preds, zero_division=0)
        if s > best_s:
            best_s, best_t = s, t
    return best_t

def split_data(df):
    X = df.drop(columns=[TARGET_COL]).select_dtypes(include=np.number).fillna(0)
    y = df[TARGET_COL]
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=TEST_RATIO, stratify=y, random_state=RANDOM_STATE)
    val_rel = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=val_rel, stratify=y_tv, random_state=RANDOM_STATE)
    return X_train, X_val, X_test, y_train, y_val, y_test

def remove_hi_leaks(X_train, X_val, X_test, domain_name):
    """
    Drop features derived from billing amounts for HI to avoid circular evaluation.
    The fraud labels in NB01 were created from Service/Medical/Total vs Approved_Cost,
    so keeping those columns in the feature set would make the task trivial and invalid.
    """
    if domain_name != 'Health Insurance':
        return X_train, X_val, X_test
    leak_kw = ['dev_','overbill','billing_variab','drg_fraud_rate',
               'max_deviation','multi_field','is_overbill',
               'amount_deviation','amount_zscore','is_large_trans',
               'is_micro_trans','is_high_value','is_service_dom']
    exact   = ['Total','Service','Medical','Approved_Cost',
                'service_to_total_ratio','NHIS ID No','Folder No','No.']
    drop = list(dict.fromkeys(
        [c for c in X_train.columns if any(k in c.lower() for k in leak_kw)] +
        [c for c in X_train.columns if c in exact]
    ))
    if drop:
        print(f'  HI: removing {len(drop)} leaking features')
        X_train = X_train.drop(columns=drop)
        X_val   = X_val.drop(columns=drop)
        X_test  = X_test.drop(columns=drop)
    if X_train.shape[1] == 0:
        raise ValueError('All HI features removed  -  check column names')
    return X_train, X_val, X_test

def apply_smote(X_train, y_train):
    fraud_rate = y_train.mean()
    if   fraud_rate < 0.01: desired = 0.05
    elif fraud_rate < 0.05: desired = 0.10
    elif fraud_rate < 0.30: desired = 0.20
    else:
        print(f'  Fraud {fraud_rate:.2%} >= 30%  -  SMOTE skipped')
        return X_train, y_train
    k  = min(5, int(y_train.sum()) - 1)
    sm = SMOTE(sampling_strategy=desired, random_state=RANDOM_STATE, k_neighbors=k)
    X_r, y_r = sm.fit_resample(X_train, y_train)
    print(f'  SMOTE: {len(X_train):,} -> {len(X_r):,}  (fraud {y_r.mean():.2%})')
    return X_r, y_r

def evaluate(y_true, y_pred, y_prob, model_name, domain):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    m = {
        'model': model_name, 'domain': domain,
        'precision': rep['1']['precision'], 'recall': rep['1']['recall'],
        'f1': rep['1']['f1-score'], 'accuracy': rep['accuracy'],
        'mcc': matthews_corrcoef(y_true, y_pred),
    }
    if y_prob is not None:
        try:
            m['roc_auc'] = roc_auc_score(y_true, y_prob)
            prec, rec, _ = precision_recall_curve(y_true, y_prob)
            m['pr_auc']  = auc(rec, prec)
        except Exception:
            m['roc_auc'] = m['pr_auc'] = 0.0
    print(f'  {model_name} [{domain}]: F1={m["f1"]:.4f} MCC={m["mcc"]:.4f} AUC={m.get("roc_auc",0):.4f}')
    return m

print('Helpers defined')

Helpers defined


## 2. Autoencoder

In [3]:
class FraudAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32, 16]):
        super().__init__()
        enc, prev = [], input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        self.encoder = nn.Sequential(*enc)
        dec = []
        for h in reversed(hidden_dims[:-1]):
            dec += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        dec.append(nn.Linear(prev, input_dim))
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def reconstruction_error(self, x_np):
        self.eval()
        with torch.no_grad():
            x = torch.FloatTensor(x_np)
            return ((x - self(x)) ** 2).mean(dim=1).numpy()

def train_autoencoder(X_legit, input_dim, domain_name):
    model = FraudAutoencoder(input_dim)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.FloatTensor(X_legit.values)),
        batch_size=256, shuffle=True
    )
    for _ in range(30):
        model.train()
        for (batch,) in loader:
            opt.zero_grad()
            loss_fn(model(batch), batch).backward()
            opt.step()
    print(f'  Autoencoder trained ({domain_name})')
    return model

print('Autoencoder defined')

Autoencoder defined


## 3. Train All Models

In [ ]:
all_results = []
os.makedirs(MODEL_DIR, exist_ok=True)

for domain_name, df in [('Credit Card', cc), ('Mobile Money', mm), ('Health Insurance', hi)]:
    print(f'\n{"="*50}\n  {domain_name}\n{"="*50}')

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)
    X_train, X_val, X_test = remove_hi_leaks(X_train, X_val, X_test, domain_name)
    print(f'  Train:{len(X_train):,} Val:{len(X_val):,} Test:{len(X_test):,} | Features:{X_train.shape[1]}')

    X_tr_sm, y_tr_sm = apply_smote(X_train, y_train)

    # Tighter regularisation for HI (1200 samples  -  easy to overfit)
    if domain_name == 'Health Insurance':
        rf_p  = dict(n_estimators=50, max_depth=4, min_samples_leaf=10,
                     min_samples_split=20, max_features='sqrt',
                     class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
        xgb_p = {**XGB_PARAMS,
                 'max_depth': 3, 'n_estimators': 50,
                 'min_child_weight': 10, 'subsample': 0.7, 'colsample_bytree': 0.7,
                 'reg_alpha': 1.0, 'reg_lambda': 2.0}
    else:
        rf_p  = RF_PARAMS
        xgb_p = XGB_PARAMS

    key = domain_name.replace(' ', '_').lower()

    # [1] Logistic Regression
    print('  [1] Logistic Regression')
    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
    lr.fit(X_tr_sm, y_tr_sm)
    y_prob = lr.predict_proba(X_test)[:,1]
    thresh = find_best_threshold(lr.predict_proba(X_val)[:,1], y_val)
    all_results.append(evaluate(y_test, (y_prob >= thresh).astype(int), y_prob, 'Logistic Regression', domain_name))
    joblib.dump(lr, os.path.join(MODEL_DIR, f'lr_{key}.pkl'))

    # [2] Random Forest
    print('  [2] Random Forest')
    rf = RandomForestClassifier(**rf_p)
    rf.fit(X_tr_sm, y_tr_sm)
    y_prob = rf.predict_proba(X_test)[:,1]
    thresh = find_best_threshold(rf.predict_proba(X_val)[:,1], y_val)
    all_results.append(evaluate(y_test, (y_prob >= thresh).astype(int), y_prob, 'Random Forest', domain_name))
    joblib.dump(rf, os.path.join(MODEL_DIR, f'rf_{key}.pkl'))

    # [3] XGBoost
    print('  [3] XGBoost')
    scale_pw = (y_tr_sm == 0).sum() / max((y_tr_sm == 1).sum(), 1)
    xgb_params = {**xgb_p, 'scale_pos_weight': scale_pw}
    xgb_params.pop('use_label_encoder', None)
    xm = xgb.XGBClassifier(**xgb_params)
    xm.fit(X_tr_sm, y_tr_sm, eval_set=[(X_val, y_val)], verbose=False)
    y_prob = xm.predict_proba(X_test)[:,1]
    thresh = find_best_threshold(xm.predict_proba(X_val)[:,1], y_val)
    all_results.append(evaluate(y_test, (y_prob >= thresh).astype(int), y_prob, 'XGBoost', domain_name))
    joblib.dump(xm, os.path.join(MODEL_DIR, f'xgb_{key}.pkl'))

    # [4] Isolation Forest
    print('  [4] Isolation Forest')
    contamination = min(float(y_train.mean()), 0.45)
    iso = IsolationForest(**ISOLATION_FOREST_PARAMS, contamination=contamination)
    iso.fit(X_train)
    raw = iso.score_samples(X_test)
    val_raw = iso.score_samples(X_val)
    s_min = min(val_raw.min(), raw.min())
    s_max = max(val_raw.max(), raw.max()) + 1e-8
    test_norm = (raw.max() - raw) / (s_max - s_min)
    val_norm  = (val_raw.max() - val_raw) / (s_max - s_min)
    thresh = find_best_threshold(val_norm, y_val)
    all_results.append(evaluate(y_test, (test_norm >= thresh).astype(int), test_norm, 'Isolation Forest', domain_name))
    joblib.dump(iso, os.path.join(MODEL_DIR, f'iso_{key}.pkl'))

    # [5] Autoencoder
    print('  [5] Autoencoder')
    ae = train_autoencoder(X_train[y_train == 0], X_train.shape[1], domain_name)
    err_raw = ae.reconstruction_error(X_test.values)
    val_raw = ae.reconstruction_error(X_val.values)
    e_min = min(val_raw.min(), err_raw.min())
    e_max = max(val_raw.max(), err_raw.max()) + 1e-8
    err_norm = (err_raw - e_min) / (e_max - e_min)
    val_norm = (val_raw - e_min) / (e_max - e_min)
    thresh   = find_best_threshold(val_norm, y_val)
    all_results.append(evaluate(y_test, (err_norm >= thresh).astype(int), err_norm, 'Autoencoder', domain_name))
    torch.save(ae.state_dict(), os.path.join(MODEL_DIR, f'ae_{key}.pt'))

print('\nAll models trained')


  Credit Card
  Train:1,296,675 Val:277,859 Test:277,860 | Features:14
  SMOTE: 1,296,675 -> 1,354,416  (fraud 4.76%)
  [1] Logistic Regression
  Logistic Regression [Credit Card]: F1=0.3224 MCC=0.3355 AUC=0.8681
  [2] Random Forest


## 4. Results

In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(OUTPUT_DIR, 'baseline_results.csv'), index=False)

for domain in ['Credit Card', 'Mobile Money', 'Health Insurance']:
    print(f'\n--- {domain} ---')
    sub = results_df[results_df['domain'] == domain]
    cols = [c for c in ['model','f1','precision','recall','mcc','roc_auc','pr_auc'] if c in sub.columns]
    print(sub[cols].sort_values('mcc', ascending=False).to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# Primary metric differs by domain:
# CC    -> ROC-AUC (0.52% fraud rate makes F1 unreliable)
# MM    -> F1      (near-balanced dataset)
# HI    -> MCC     (~40% fraud, F1 can look inflated for weak models)

DOMAIN_CONFIG = {
    'Credit Card':      {'metric': 'roc_auc', 'label': 'ROC-AUC', 'color': '#1565C0'},
    'Mobile Money':     {'metric': 'f1',       'label': 'F1',      'color': '#2E7D32'},
    'Health Insurance': {'metric': 'mcc',      'label': 'MCC',     'color': '#E65100'},
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (domain, cfg) in zip(axes, DOMAIN_CONFIG.items()):
    sub  = results_df[results_df['domain'] == domain].sort_values(cfg['metric'])
    bars = ax.barh(sub['model'], sub[cfg['metric']], color=cfg['color'], alpha=0.82)
    for bar, val in zip(bars, sub[cfg['metric']]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9, fontweight='bold')
    ax.set_xlim(0, 1.12)
    ax.set_title(domain, fontweight='bold')
    ax.set_xlabel(cfg['label'])
    ax.axvline(0.8, color='red', linestyle='--', alpha=0.4)

plt.suptitle('Baseline Performance  -  Primary Metric per Domain', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'baseline_results.png'), dpi=150, bbox_inches='tight')
plt.show()